# Volume 7, Chapter 10: Tauberian Theorems (Python companion)

Summation methods, Karamata in action on quicksort and on sums of two squares, and the prime number theorem watched converging.

*Before each solution, predict what the output will look like.*

## Exercises

**P10.1 (★) The three summation methods, side by side.** For each of $\sum(-1)^n$, $\sum(-1)^n(n+1)$, $\sum\frac{(-1)^n}{\sqrt{n+1}}$ and $\sum\frac{(-1)^n}{n+1}$, compute the partial sums $s_n$, the Cesàro means $\sigma_n$ and the Abel means $f(x)$ at $x=1-\frac1{n+1}$, for $n$ up to $10^4$. (a) Which of the four converge, which are Cesàro-summable, which are Abel-summable? (b) Check Tauber's condition $na_n\to0$ in each case, and verify that it predicts exactly which Abel limits are genuine sums. (c) For $\sum\frac{(-1)^n}{n+1}$, how fast do the three methods approach $\log 2$? Plot the errors on a log-log scale and explain the slopes.

**P10.2 (★★) CS application: quicksort, predicted by Karamata.** (a) Implement quicksort with a uniformly random pivot and count comparisons; average over many random inputs for $n=2^4,\dots,2^{13}$. (b) Compute the exact $C_n$ from the recurrence $C_n=n-1+\frac2n\sum_{k<n}C_k$ and check it against the closed form $2(n+1)H_n-4n$ and against the measurements. (c) Exercise 10.4 deduced $C_n\sim2n\log n$ from the singularity of $C(z)=\frac{2}{(1-z)^2}\bigl(\log\frac1{1-z}-z\bigr)$. Plot $\frac{C_n}{2n\log n}$ and see how slowly it approaches $1$; where does the missing $-4n$ show up? Compare with the information-theoretic lower bound $\log_2(n!)$.

**P10.3 (★★) The prime number theorem, and how slowly it converges.** (a) Sieve to $10^7$ and plot $\frac{\psi(x)}x$, $\frac{\theta(x)}x$ and $\frac{\pi(x)\log x}x$. (b) Corollary 10.15 gives $\psi(x;4,a)\sim\frac x2$ for $a=1,3$. Plot $\frac{2\psi(x;4,a)}x$ for both classes on the same axes: which one is ahead, and for how long? (c) The Tauberian proof gives no error term. Compare the observed error with $\frac{\sqrt x\log^2x}x$, the size the Riemann hypothesis would give, and with $\exp(-c\sqrt{\log x})$, the size the classical zero-free region gives.

**P10.4 (★★) The Landau--Ramanujan constant.** (a) Compute $K=\frac1{\sqrt2}\prod_{q\equiv3(4)}(1-q^{-2})^{-1/2}$ from the primes up to $10^6$ and estimate the error of truncation. (b) Mark the integers up to $10^8$ that are sums of two squares (use the criterion of Volume 1, Theorem 7.3 with a sieve, not a search over $x^2+y^2$), and compare $\sum_{n\le x,\,n=\square+\square}\frac1n$ with the prediction $2K\sqrt{\log x}$ of Theorem 10.17. (c) Compare $B(x)\frac{\sqrt{\log x}}x$ with $K$ (Landau's unweighted theorem, which we did not prove). Which of the two converges faster, and why does the weighted one deserve its reputation for being easier?

## Solutions

### P10.1

In [ ]:
import numpy as np, math
import matplotlib.pyplot as plt

N = 10_000
n = np.arange(N + 1)

series = {
    r"$\sum(-1)^n$":                 (-1.0) ** n,
    r"$\sum(-1)^n(n+1)$":            (-1.0) ** n * (n + 1),
    r"$\sum(-1)^n/\sqrt{n+1}$":      (-1.0) ** n / np.sqrt(n + 1),
    r"$\sum(-1)^n/(n+1)$":           (-1.0) ** n / (n + 1),
}

# The Abel means are evaluated at x = 0.99, where the tail 0.99^10000 is about 1e-44, so the
# truncated sum is the true f(x) to machine precision. At x = 1 - 1/N the truncation error
# would swamp everything, which is itself worth seeing (last line).
X = 0.99

for name, a in series.items():
    s = np.cumsum(a)
    sig = np.cumsum(s) / (n + 1)
    f = np.polyval(a[::-1], X)
    print(f"{name:28s}  s_N = {s[-1]:12.6f}   sigma_N = {sig[-1]:12.6f}   f(0.99) = {f:12.6f}"
          f"   n a_n = {n[-1]*a[-1]:10.4f}")
print()
print("exact Abel limits: 0.5, 0.25, 0.604899..., log 2 = 0.693147...")
print("truncated at N, the Abel mean of the first series at x = 1-1/N is useless:",
      np.polyval(series[list(series)[0]][::-1], 1 - 1 / (N + 1)))

(a) $\sum(-1)^n$ does not converge, is Cesàro-summable to $\frac12$ and Abel-summable to $\frac12$. $\sum(-1)^n(n+1)$ is neither convergent nor Cesàro-summable (the means oscillate between $0$ and $\frac12$), but is Abel-summable to $\frac14$: Abel's method is strictly stronger, as Exercise 10.1 shows. $\sum\frac{(-1)^n}{\sqrt{n+1}}$ and $\sum\frac{(-1)^n}{n+1}$ converge, so all three methods agree, by Theorem 10.2 and Corollary 10.3.

(b) $na_n\to0$ holds only for the last two, which are exactly the two whose Abel limit is an honest sum. For $\sum(-1)^n$ we have $na_n=\pm n$, and for $\sum\frac{(-1)^n}{\sqrt{n+1}}$ we have $na_n\approx\pm\sqrt n$: Tauber's hypothesis fails for the third one too, yet the series converges. The condition is sufficient, not necessary --- and Littlewood's $a_n=O(1/n)$, which does hold for the fourth, is the sharp version.

In [ ]:
a = series[r"$\sum(-1)^n/(n+1)$"]
s = np.cumsum(a); sig = np.cumsum(s) / (n + 1)
target = np.log(2)
idx = np.unique(np.geomspace(2, N, 60).astype(int))
x = 1 - 1 / (idx + 1)
fab = np.log1p(x) / x          # the exact Abel mean of this series: f(x) = log(1+x)/x

plt.figure(figsize=(6.5, 4))
plt.loglog(idx, np.abs(s[idx] - target), label="partial sums")
plt.loglog(idx, np.abs(sig[idx] - target), label=r"Ces\`aro means")
plt.loglog(idx, np.abs(fab - target), label="Abel means")
plt.loglog(idx, 0.5 / idx, "k:", label=r"$1/(2n)$")
plt.xlabel("n"); plt.ylabel("error"); plt.legend(); plt.title("Three ways to reach log 2")
plt.show()
for k in (10, 100, 1000, 10000):
    xk = 1 - 1 / (k + 1)
    print(f"n={k:6d}  s={s[k]-target:+.3e}  sigma={sig[k]-target:+.3e}  abel={math.log1p(xk)/xk-target:+.3e}")

(c) All three errors are of order $\frac1n$, with different constants: the partial sums alternate around $\log2$ with error $\approx\frac{0.50}n$, while the Cesàro and Abel means both come in at $\approx\frac{0.193}n$ (for the Abel mean the constant is $-\frac{\mathrm d}{\mathrm dx}\frac{\log(1+x)}x$ at $x=1$, which is $\log2-\frac12=0.193$). Averaging the partial sums kills the alternation and buys a constant, not a rate --- another instance of the rule that these methods never produce error terms.

### P10.2

In [ ]:
import numpy as np, random, math
import matplotlib.pyplot as plt

def quicksort_comparisons(a):
    """Comparisons made by quicksort with a uniformly random pivot (iterative, to avoid recursion limits)."""
    total = 0
    stack = [a]
    while stack:
        part = stack.pop()
        if len(part) < 2:
            continue
        p = part[random.randrange(len(part))]
        lo, hi, eq = [], [], 0
        for v in part:
            if v < p: lo.append(v)
            elif v > p: hi.append(v)
            else: eq += 1
        total += len(part) - 1          # every other element is compared with the pivot
        stack.append(lo); stack.append(hi)
    return total

sizes = [2 ** k for k in range(4, 14)]
measured = []
for nn in sizes:
    reps = max(5, 2000 // nn + 5)
    arr = list(range(nn))
    measured.append(np.mean([quicksort_comparisons(random.sample(arr, nn)) for _ in range(reps)]))

N = 20_000           # exact recurrence, beyond the measured range
C = np.zeros(N + 1); run = 0.0
for k in range(1, N + 1):
    C[k] = k - 1 + 2 * run / k
    run += C[k]
closed = np.array([2 * (m + 1) * sum(1 / j for j in range(1, m + 1)) - 4 * m if m else 0.0 for m in sizes])

print(f"{'n':>7} {'measured':>12} {'exact C_n':>12} {'closed form':>12} {'2 n log n':>12} {'log2(n!)':>12}")
for nn, mm, cf in zip(sizes, measured, closed):
    print(f"{nn:7d} {mm:12.1f} {C[nn]:12.1f} {cf:12.1f} {2*nn*math.log(nn):12.1f} {math.lgamma(nn+1)/math.log(2):12.1f}")

(a)--(b) The measured averages sit on the exact $C_n$ to within a percent or two, which is the sampling noise of a handful of random inputs, and the closed form $2(n+1)H_n-4n$ reproduces the recurrence exactly.

In [ ]:
ns = np.arange(16, N + 1)
plt.figure(figsize=(6.5, 4))
plt.semilogx(ns, C[16:] / (2 * ns * np.log(ns)), label=r"$C_n/(2n\log n)$")
plt.semilogx(ns, C[16:] / (2 * ns * (np.log(ns) + np.euler_gamma - 2)), label=r"$C_n/(2n(\log n+\gamma-2))$")
plt.axhline(1, color="k", lw=0.8)
plt.xlabel("n"); plt.legend(); plt.title("How slowly the Tauberian asymptotic arrives")
plt.show()
for nn in (10**2, 10**3, 10**4):
    print(f"n={nn:6d}  C_n/(2n log n) = {C[nn]/(2*nn*math.log(nn)):.4f}   "
          f"ratio to log2(n!) = {C[nn]/(math.lgamma(nn+1)/math.log(2)):.4f}")

(c) $\frac{C_n}{2n\log n}$ is still only about $0.85$ at $n=10^4$ and crawls upwards: the exact value is $C_n=2n\log n+(2\gamma-4)n+O(\log n)$, and the linear term $(2\gamma-4)n\approx-2.85n$ is a relative correction of size $\frac{1.4}{\log n}$, which is $15\%$ at $n=10^4$ and still $7\%$ at $n=10^9$. Replacing $\log n$ by $\log n+\gamma-2$ makes the ratio flat at once. This is the price of a Tauberian theorem: it gets the leading term with almost no information, and says nothing at all about the next one. Against the information-theoretic bound $\log_2(n!)\approx1.44\,n\log n$, quicksort costs about $38\%$ more comparisons in the limit.

### P10.3

In [ ]:
import numpy as np, math
import matplotlib.pyplot as plt

LIM = 10 ** 7

def sieve_mask(limit):
    s = np.ones(limit + 1, dtype=bool); s[:2] = False
    for p in range(2, int(limit ** 0.5) + 1):
        if s[p]: s[p*p::p] = False
    return s

def sieve(limit):
    return np.flatnonzero(sieve_mask(limit))

is_prime = sieve_mask(LIM)
primes = np.flatnonzero(is_prime)
print(f"{len(primes)} primes up to {LIM}")

# von Mangoldt weights
lam = np.zeros(LIM + 1)
lam[primes] = np.log(primes)
for p in primes[primes <= int(LIM ** 0.5)]:
    q = p * p
    while q <= LIM:
        lam[q] = math.log(p); q *= p

psi = np.cumsum(lam)
theta = np.cumsum(np.where(is_prime, np.log(np.maximum(np.arange(LIM + 1), 2)), 0.0))
pi = np.cumsum(is_prime)

xs = np.unique(np.geomspace(10, LIM, 300).astype(int))
plt.figure(figsize=(6.5, 4))
plt.semilogx(xs, psi[xs] / xs, label=r"$\psi(x)/x$")
plt.semilogx(xs, theta[xs] / xs, label=r"$\theta(x)/x$")
plt.semilogx(xs, pi[xs] * np.log(xs) / xs, label=r"$\pi(x)\log x/x$")
plt.axhline(1, color="k", lw=0.8); plt.legend(); plt.xlabel("x")
plt.title("The prime number theorem, converging")
plt.show()
for x in (10**3, 10**4, 10**5, 10**6, 10**7):
    print(f"x={x:9d}  psi/x={psi[x]/x:.5f}  theta/x={theta[x]/x:.5f}  pi log x/x={pi[x]*math.log(x)/x:.5f}")

(a) $\frac{\psi(x)}x$ is within $1.5\cdot10^{-4}$ of $1$ at $x=10^7$, $\frac{\theta(x)}x$ slightly below it (the difference is the prime powers, of size $\approx2\sqrt x$), and $\frac{\pi(x)\log x}x$ is still near $1.07$: the approximation $\frac x{\log x}$ is much worse than $\operatorname{li}(x)$, since $\operatorname{li}(x)=\frac x{\log x}(1+\frac1{\log x}+\dots)$ and $\frac1{\log x}\approx0.062$ here.

In [ ]:
res = np.arange(LIM + 1) % 4
psi1 = np.cumsum(np.where(res == 1, lam, 0.0))
psi3 = np.cumsum(np.where(res == 3, lam, 0.0))

plt.figure(figsize=(6.5, 4))
plt.semilogx(xs, 2 * psi1[xs] / xs, label=r"$2\psi(x;4,1)/x$")
plt.semilogx(xs, 2 * psi3[xs] / xs, label=r"$2\psi(x;4,3)/x$")
plt.axhline(1, color="k", lw=0.8); plt.legend(); plt.xlabel("x")
plt.title("Chebyshev's bias: 3 mod 4 leads")
plt.show()
for x in (10**4, 10**5, 10**6, 10**7):
    print(f"x={x:9d}  2psi(x;4,1)/x={2*psi1[x]/x:.5f}   2psi(x;4,3)/x={2*psi3[x]/x:.5f}")
print()
pi1 = np.cumsum(is_prime & (res == 1)); pi3 = np.cumsum(is_prime & (res == 3))
for x in (10**4, 10**5, 10**6, 10**7):
    print(f"x={x:9d}  pi(x;4,1)={pi1[x]:8d}   pi(x;4,3)={pi3[x]:8d}   lead of 3 mod 4 = {pi3[x]-pi1[x]:6d}")
print()
for x in (10**4, 10**5, 10**6, 10**7):
    err = abs(psi[x] / x - 1)
    print(f"x={x:9d}  |psi(x)/x-1|={err:.3e}   RH size sqrt(x)log^2(x)/x={math.sqrt(x)*math.log(x)**2/x:.3e}"
          f"   zero-free region exp(-sqrt(log x))={math.exp(-math.sqrt(math.log(x))):.3e}")

(b) Both classes tend to $\frac12$, as Corollary 10.15 says, and by $x=10^7$ they agree to four decimals. The interesting part is which one leads. Counted by $\pi$, the class $3\bmod4$ is ahead at every $x$ in the range: this is Chebyshev's bias of 1853. Counted by $\psi$ the lead changes hands, because $\psi$ also counts the prime squares, and $p^2\equiv1\pmod4$ for every odd $p$; there are about $2\sqrt x$ of them, all in the class $1$, which is the size of the bias. Neither fact is visible to the Tauberian theorem, which sees only the limit.

(c) The observed error is comfortably below the Riemann-hypothesis size $\sqrt x\log^2x$ and far below $\exp(-c\sqrt{\log x})$, which at $x=10^7$ is still about $1.5\%$ --- the classical zero-free region is a very weak statement numerically, and the Tauberian proof gives no bound at all. Theorem 10.13 is a limit theorem: it promises the ratio tends to $1$ and never says when.

### P10.4

In [ ]:
import numpy as np, math
import matplotlib.pyplot as plt

P = 10 ** 6
small = sieve(P)
q3 = small[small % 4 == 3]
logK = -0.5 * math.log(2) - 0.5 * np.sum(np.log1p(-1.0 / q3.astype(float) ** 2))
K = math.exp(logK)
print(f"K from primes up to {P}: {K:.10f}")
print(f"true value (Finch):        0.7642236536")
print(f"crude bound on the truncated tail of log K: about {1/(2*P*math.log(P)):.2e}")

In [ ]:
LIM2 = 10 ** 7

# b(n) = 1 iff every prime q = 3 mod 4 divides n to an even power.
# Sieve: start with all ones, and for each q = 3 mod 4 kill the n whose q-exponent is odd.
rep = np.ones(LIM2 + 1, dtype=bool); rep[0] = False
for q in sieve(int(LIM2 ** 0.5) + 1):
    if q % 4 != 3:
        continue
    # kill multiples of q that are not multiples of q^2, then recurse upward
    e = 1
    while q ** e <= LIM2:
        if e % 2 == 1:
            step = q ** e
            idx = np.arange(step, LIM2 + 1, step)
            idx = idx[(idx // step) % q != 0]
            rep[idx] = False
        e += 1
# primes q = 3 mod 4 above sqrt(LIM2) appear to the first power only
big = sieve(LIM2)
big = big[(big % 4 == 3) & (big > int(LIM2 ** 0.5))]
for q in big:
    rep[q::q] = False
print("representable up to 100:", np.flatnonzero(rep[:101]))
print("count up to 10^7:", int(rep.sum()))

In [ ]:
w = np.where(rep, 1.0 / np.maximum(np.arange(LIM2 + 1), 1), 0.0)
S = np.cumsum(w)                      # sum_{n<=x, representable} 1/n
B = np.cumsum(rep)                    # B(x)

xs = np.unique(np.geomspace(100, LIM2, 200).astype(int))
plt.figure(figsize=(6.5, 4))
plt.semilogx(xs, S[xs] / (2 * K * np.sqrt(np.log(xs))), label=r"weighted: $\sum 1/n\ /\ 2K\sqrt{\log x}$")
plt.semilogx(xs, B[xs] * np.sqrt(np.log(xs)) / (xs * K), label=r"unweighted: $B(x)\sqrt{\log x}/(Kx)$")
plt.axhline(1, color="k", lw=0.8); plt.legend(); plt.xlabel("x")
plt.title("Theorem 10.17 (weighted) and Landau's theorem (unweighted)")
plt.show()
for x in (10**4, 10**6, 10**7):
    print(f"x={x:10d}  sum 1/n = {S[x]:.5f}   2K sqrt(log x) = {2*K*math.sqrt(math.log(x)):.5f}"
          f"   ratio = {S[x]/(2*K*math.sqrt(math.log(x))):.5f}")
print()
for x in (10**4, 10**6, 10**7):
    print(f"x={x:10d}  B(x) = {B[x]:10d}   Kx/sqrt(log x) = {K*x/math.sqrt(math.log(x)):12.1f}"
          f"   ratio = {B[x]*math.sqrt(math.log(x))/(K*x):.5f}")

(a) The product over the primes $q\equiv3$ up to $10^6$ gives $K=0.76422364$, about $1.3\cdot10^{-8}$ below the true $0.7642236536$, in line with the crude tail bound. (Many more digits need an acceleration that writes the product in terms of values of $\zeta$ and $L(\cdot,\chi_4)$ at integers.)

(b)--(c) The two statements are asymptotically equivalent in content but not in numerical behaviour, and the correction in each is of relative size $O(\frac1{\log x})$. At $x=10^7$ the weighted ratio is $0.994$ and rising towards $1$ from below, while the unweighted one is $1.043$ and falling towards $1$ from above, having been $1.09$ at $x=10^4$. The weighted sum is far the smoother of the two: summing $\frac1n$ averages over all scales below $x$, so it has no fluctuation left, while $B(x)$ still feels the local irregularity of the representable integers. That smoothing is exactly what made Theorem 10.17 reachable from Karamata's theorem in half a page, while the unweighted statement needed Landau's contour analysis around a branch point.